# Etapa 9 — Dumb Baseline (sem sentimento)

## Por que isto existe

Toda a narrativa do TCC depende da afirmação:

> *5 features de sentimento FinBERT-PT-BR fazem o Transformer atingir AUC = 0,709, superando os 1.024 embeddings genéricos da Etapa 3.*

Mas falta um número crítico: **qual é o AUC de um modelo que não usa sentimento nenhum, apenas features triviais de preço?**

Sem esse número, o resultado de 0,709 fica suspenso no ar. Ele pode estar:

- **Muito acima** de um baseline puramente autoregressivo → o sentimento adiciona valor real (bom)
- **Muito próximo** do baseline autoregressivo → o sentimento é decoração; o sinal vem dos lags de preço (ruim)
- **Abaixo** do baseline autoregressivo → o sentimento adiciona ruído (péssimo)

Este notebook produz esse número.

## Protocolo

- **Features**: apenas 5, todas autoregressivas, **nenhuma proveniente de notícias** — `return`, `lag_1`, `lag_5`, `volume`, `std21`.
- **Target**: idêntico ao da Etapa 4 — `1` se `Close[t+21] > Close[t]`, senão `0`.
- **Split**: walk-forward 70/15/15, mesmo split usado em todas as etapas anteriores.
- **Modelos**: 4 modelos para garantir comparabilidade direta com a Etapa 4 — Logistic Regression (sanity), XGBoost, BiLSTM reduzido, Transformer pequeno.
- **Métricas**: ROC-AUC com **intervalo de confiança bootstrap 95%** (1.000 reamostragens).

Todas as utilidades de avaliação vivem em `eval_utils.py` para serem reusadas pelos próximos experimentos (multi-ticker, regime split).

## 1. Imports e carregamento do dataset

In [1]:
import sys, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from eval_utils import (
    walk_forward_split, evaluate_model, make_binary_target,
    format_metric_with_ci,
)

DATASET_PATH = '../2.stocks/dataset_full.csv'
HORIZON = 21
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

In [2]:
df = pd.read_csv(DATASET_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)

# Construir target binário com horizonte de 21 dias
df['target'] = make_binary_target(df['Close'], horizon=HORIZON)
df = df.dropna(subset=['target']).reset_index(drop=True)

# As 5 features autoregressivas — NENHUMA derivada de notícias
FEATURES = ['return', 'lag_1', 'lag_5', 'Volume', 'std21']
print(f'Dataset: {len(df)} dias | balance sobe={df["target"].mean():.3f}')
print(f'Features: {FEATURES}')

Dataset: 1206 dias | balance sobe=0.590
Features: ['return', 'lag_1', 'lag_5', 'Volume', 'std21']


## 2. Split walk-forward e normalização

O `StandardScaler` é fitado **apenas no treino** para evitar leakage.

In [3]:
train_df, val_df, test_df = walk_forward_split(df, train_frac=0.70, val_frac=0.15)
print(f'Train: {len(train_df)} ({train_df["Date"].min().date()} → {train_df["Date"].max().date()})')
print(f'Val:   {len(val_df)} ({val_df["Date"].min().date()} → {val_df["Date"].max().date()})')
print(f'Test:  {len(test_df)} ({test_df["Date"].min().date()} → {test_df["Date"].max().date()})')

scaler = StandardScaler().fit(train_df[FEATURES])
X_train = scaler.transform(train_df[FEATURES]); y_train = train_df['target'].values.astype(int)
X_val   = scaler.transform(val_df[FEATURES]);   y_val   = val_df['target'].values.astype(int)
X_test  = scaler.transform(test_df[FEATURES]);  y_test  = test_df['target'].values.astype(int)

Train: 844 (2021-04-28 → 2024-09-10)
Val:   180 (2024-09-11 → 2025-06-04)
Test:  182 (2025-06-05 → 2026-02-25)


## 3. Modelo A — Logistic Regression (sanity check)

Antes de treinar nada complexo, um regressor logístico responde à pergunta:
*as 5 features triviais têm relação linear com o target?*

In [4]:
lr = LogisticRegression(max_iter=2000, class_weight='balanced').fit(X_train, y_train)
y_score = lr.predict_proba(X_test)[:, 1]
results_lr = evaluate_model(y_test, y_score)
print('Logistic Regression — AUC:', results_lr['auc_str'])
print(results_lr['report'])

Logistic Regression — AUC: 0.415 [0.311, 0.530]
              precision    recall  f1-score   support

           0       0.31      1.00      0.47        56
           1       0.00      0.00      0.00       126

    accuracy                           0.31       182
   macro avg       0.15      0.50      0.24       182
weighted avg       0.09      0.31      0.14       182



## 4. Modelo B — XGBoost (não-linear, comparável à Etapa 4)

In [5]:
pos = (y_train == 1).sum(); neg = (y_train == 0).sum()
scale_pos_weight = neg / max(pos, 1)

xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc', random_state=SEED,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
y_score = xgb_model.predict_proba(X_test)[:, 1]
results_xgb = evaluate_model(y_test, y_score)
print('XGBoost — AUC:', results_xgb['auc_str'])
print(results_xgb['report'])

XGBoost — AUC: 0.658 [0.565, 0.744]
              precision    recall  f1-score   support

           0       0.31      1.00      0.47        56
           1       1.00      0.02      0.03       126

    accuracy                           0.32       182
   macro avg       0.66      0.51      0.25       182
weighted avg       0.79      0.32      0.17       182



## 5. Modelo C — BiLSTM reduzido (sequencial)

Janelas de 30 dias. Mesma arquitetura reduzida da Etapa 3, treinada do zero.

In [6]:
WINDOW = 30

def make_windows(X, y, window=WINDOW):
    Xs, ys = [], []
    for i in range(window, len(X)):
        Xs.append(X[i-window:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

Xw_train, yw_train = make_windows(X_train, y_train)
Xw_val,   yw_val   = make_windows(X_val,   y_val)
Xw_test,  yw_test  = make_windows(X_test,  y_test)
print('Shapes:', Xw_train.shape, Xw_val.shape, Xw_test.shape)

class BiLSTMSmall(nn.Module):
    def __init__(self, n_features, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=1, batch_first=True, bidirectional=True, dropout=0.0)
        self.fc = nn.Sequential(nn.Linear(hidden*2, 16), nn.ReLU(), nn.Dropout(0.3), nn.Linear(16, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = BiLSTMSmall(n_features=len(FEATURES)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
pos_weight = torch.tensor([neg / max(pos, 1)], device=device, dtype=torch.float32)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

Xt = torch.tensor(Xw_train, dtype=torch.float32, device=device)
yt = torch.tensor(yw_train, dtype=torch.float32, device=device)
Xv = torch.tensor(Xw_val,   dtype=torch.float32, device=device)
yv = torch.tensor(yw_val,   dtype=torch.float32, device=device)
Xte = torch.tensor(Xw_test, dtype=torch.float32, device=device)

best_val = float('inf'); best_state = None; patience = 10; bad = 0
for epoch in range(200):
    model.train(); opt.zero_grad()
    logits = model(Xt); loss = loss_fn(logits, yt)
    loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(Xv), yv).item()
    if val_loss < best_val - 1e-4:
        best_val = val_loss; best_state = {k: v.clone() for k, v in model.state_dict().items()}; bad = 0
    else:
        bad += 1
        if bad >= patience: break
model.load_state_dict(best_state); model.eval()
with torch.no_grad():
    y_score = torch.sigmoid(model(Xte)).cpu().numpy()
results_lstm = evaluate_model(yw_test, y_score)
print('BiLSTM small — AUC:', results_lstm['auc_str'])
print(results_lstm['report'])

Shapes: (814, 30, 5) (150, 30, 5) (152, 30, 5)
BiLSTM small — AUC: 0.302 [0.196, 0.420]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        36
           1       0.76      1.00      0.87       116

    accuracy                           0.76       152
   macro avg       0.38      0.50      0.43       152
weighted avg       0.58      0.76      0.66       152



## 6. Modelo D — Transformer pequeno

Mesma arquitetura conceitual da Etapa 4 (encoder + mean pooling), reduzida para 5 features.

In [7]:
class SmallTransformer(nn.Module):
    def __init__(self, n_features, d_model=32, nhead=4, nlayers=2):
        super().__init__()
        self.proj = nn.Linear(n_features, d_model)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=64, dropout=0.2, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=nlayers)
        self.head = nn.Sequential(nn.Linear(d_model, 16), nn.ReLU(), nn.Linear(16, 1))
    def forward(self, x):
        h = self.enc(self.proj(x))
        return self.head(h.mean(dim=1)).squeeze(-1)

model = SmallTransformer(n_features=len(FEATURES)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

best_val = float('inf'); best_state = None; bad = 0
for epoch in range(200):
    model.train(); opt.zero_grad()
    loss = loss_fn(model(Xt), yt); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(Xv), yv).item()
    if val_loss < best_val - 1e-4:
        best_val = val_loss; best_state = {k: v.clone() for k, v in model.state_dict().items()}; bad = 0
    else:
        bad += 1
        if bad >= patience: break
model.load_state_dict(best_state); model.eval()
with torch.no_grad():
    y_score = torch.sigmoid(model(Xte)).cpu().numpy()
results_tr = evaluate_model(yw_test, y_score)
print('Transformer small — AUC:', results_tr['auc_str'])
print(results_tr['report'])

Transformer small — AUC: 0.076 [0.037, 0.120]
              precision    recall  f1-score   support

           0       0.21      0.86      0.34        36
           1       0.00      0.00      0.00       116

    accuracy                           0.20       152
   macro avg       0.11      0.43      0.17       152
weighted avg       0.05      0.20      0.08       152



## 7. Tabela final — o número que reescreve a tese

Esta tabela é o **único output** que importa deste notebook. Ela define o piso contra o qual a Etapa 4 (FinBERT, AUC = 0,709) deve ser comparada.

In [9]:
summary = pd.DataFrame([
    {'modelo': 'Logistic Regression', **{k: results_lr[k]   for k in ['auc','accuracy','f1_pos','f1_neg']}, 'auc_ci': results_lr['auc_str']},
    {'modelo': 'XGBoost',             **{k: results_xgb[k]  for k in ['auc','accuracy','f1_pos','f1_neg']}, 'auc_ci': results_xgb['auc_str']},
    {'modelo': 'BiLSTM small',        **{k: results_lstm[k] for k in ['auc','accuracy','f1_pos','f1_neg']}, 'auc_ci': results_lstm['auc_str']},
    {'modelo': 'Transformer small',   **{k: results_tr[k]   for k in ['auc','accuracy','f1_pos','f1_neg']}, 'auc_ci': results_tr['auc_str']},
])
summary.to_csv('results_dumb_baseline.csv', index=False)
summary

,modelo,auc,accuracy,f1_pos,f1_neg,auc_ci
0,Logistic Regression,0.414824,0.307692,0.000000,0.470588,"0.415 [0.311, 0.530]"
1,XGBoost,0.658305,0.318681,0.031250,0.474576,"0.658 [0.565, 0.744]"
2,BiLSTM small,0.302203,0.763158,0.865672,0.000000,"0.302 [0.196, 0.420]"
3,Transformer small,0.076389,0.203947,0.000000,0.338798,"0.076 [0.037, 0.120]"


## 8. Como interpretar

Compare a coluna `auc_ci` acima com o resultado da Etapa 4: **Transformer + FinBERT = 0.709**.

Três cenários possíveis:

| Cenário | Melhor baseline AUC | Interpretação | Ação para a banca |
|---|---|---|---|
| **A** | < 0.60 | Sentimento adiciona valor real (Δ ≥ 0.10) | Reportar Δ explicitamente; é o seu hero number |
| **B** | 0.60–0.68 | Sentimento adiciona valor marginal | Reportar honestamente; comparar CIs — pode não ser significativo |
| **C** | ≥ 0.68 | A maior parte do sinal vem dos preços, não das notícias | Pivotar a tese: o achado real é a comparação entre representações textuais (Etapa 3 vs 4), não a previsão absoluta |

**Critério de significância estatística:** se o intervalo de confiança bootstrap do baseline e o da Etapa 4 **se sobrepõem**, a diferença não é estatisticamente clara — você precisa reportar isso.

Próximos passos após este notebook:
1. Reaplicar este mesmo baseline em PETR4 e VALE3 (Etapa 10 — multi-ticker).
2. Recalcular o AUC da Etapa 4 com o mesmo `bootstrap_auc_ci` para tornar as comparações simétricas.
3. Regime split do conjunto de teste por volatilidade do IBOV (Etapa 11).